In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('iris.csv')
X =  df.iloc[:, 0:4].values
y_raw = df.iloc[:, 4].values

label_map = {'setosa':0, 'versicolor':1, 'virginica': 2}
y_int = np.array([label_map[label] for label in y_raw])

def one_hot_encode(y,num_classess):
    one_hot = np.zeros((len(y), num_classess))
    one_hot[np.arange(len(y)), y] = 1
    return one_hot

y = one_hot_encode(y_int, 3)

X = (X - X.mean(axis=0)) / X.std(axis=0)

np.random.seed(42)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)

split = int(0.8 * X.shape[0])
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def d_sigmoid(x):
    return x * (1 - x)

def forward_pass(x,w1,w2,w3,b1,b2,b3):
    h1_in = np.dot(x,w1) + b1
    h1_out = sigmoid(h1_in)
    h2_in = np.dot(h1_out,w2) + b2
    h2_out = sigmoid(h2_in)
    o_in = np.dot(h2_out,w3) + b3
    o_out = sigmoid(o_in)
    return h1_out,h2_out,o_out

def do_gradient_descent():
    np.random.seed(0)
    input_neuron = 4
    hidden_neuron1 = 6  
    hidden_neuron2 = 6  
    output_neuron = 3
    
    w1 = np.random.randn(input_neuron, hidden_neuron1) * 0.1
    b1 = np.random.randn(1, hidden_neuron1) * 0.1
    w2 = np.random.randn(hidden_neuron1, hidden_neuron2) * 0.1
    b2 = np.random.randn(1, hidden_neuron2) * 0.1
    w3 = np.random.randn(hidden_neuron2, output_neuron) * 0.1
    b3 = np.random.randn(1, output_neuron) * 0.1
    
    eta = 0.1
    max_epoch = 1000 
    
    for i in range(max_epoch):
        dw1 = np.zeros_like(w1)
        dw2 = np.zeros_like(w2)
        dw3 = np.zeros_like(w3)
        db1 = np.zeros_like(b1)
        db2 = np.zeros_like(b2)
        db3 = np.zeros_like(b3)
        total_error = 0
        for x, y in zip(X_train, y_train):
            x = x.reshape(1, -1)
            y = y.reshape(1, -1)
            h1_out, h2_out, fx = forward_pass(x, w1, w2, w3, b1, b2, b3)
            total_error += 0.5 * np.mean((y - fx) ** 2)
            d_out = (fx - y) * d_sigmoid(fx)
            d_h2 = np.dot(d_out, w3.T) * d_sigmoid(h2_out)
            d_h1 = np.dot(d_h2, w2.T) * d_sigmoid(h1_out)
            dw3 += np.dot(h2_out.T, d_out)
            db3 += d_out
            dw2 += np.dot(h1_out.T, d_h2)
            db2 += d_h2
            dw1 += np.dot(x.T, d_h1)
            db1 += d_h1
        w1 -= eta * dw1
        b1 -= eta * db1
        w2 -= eta * dw2
        b2 -= eta * db2
        w3 -= eta * dw3
        b3 -= eta * db3

    print(f"Epoch {i+1}, Mean Squared Error: {total_error/len(X_train):.6f}")
    
    return w1, w2, w3, b1, b2, b3
    np.savez(
    "iris_model.npz",
    w1=w1, w2=w2, w3=w3,
    b1=b1, b2=b2, b3=b3,
    X_mean=X.mean(axis=0),
    X_std=X.std(axis=0)
)


w1, w2, w3, b1, b2, b3 = do_gradient_descent()

# Function to predict class from user input
def predict_from_input(w1, w2, w3, b1, b2, b3):
    species_names = ["setosa", "versicolor", "virginica"]
    while True:
        try:
            # Get user input for the four features
            print("\nEnter Iris features (sepal_length sepal_width petal_length petal_width), or 'quit' to exit:")
            user_input = input().lower().strip()
            if user_input == 'quit':
                print("Exiting prediction mode.")
                break
            
            # Split and convert to float
            features = [float(x) for x in user_input.split()]
            if len(features) != 4:
                print("Please enter exactly 4 values separated by spaces.")
                continue
            
            # Normalize input using training data stats
            x = np.array(features).reshape(1, -1)
            x_mean = np.mean(X, axis=0)
            x_std = np.std(X, axis=0)
            x_normalized = (x - x_mean) / x_std
            
            # Perform forward pass
            _, _, fx = forward_pass(x_normalized, w1, w2, w3, b1, b2, b3)
            predicted_class = np.argmax(fx)
            
            # Output prediction
            print(f"Predicted species: {species_names[predicted_class]}")
            
        except ValueError:
            print("Please enter valid numerical values separated by spaces.")
        except Exception as e:
            print(f"An error occurred: {e}")

# Run prediction with user input
predict_from_input(w1, w2, w3, b1, b2, b3)

Epoch 1000, Mean Squared Error: 0.003121

Enter Iris features (sepal_length sepal_width petal_length petal_width), or 'quit' to exit:
